<a href="https://colab.research.google.com/github/mohamedalangr/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
#### We are auditing two primary signals that our baseline refresh scoring rule will rely on:


1.   Signal 1: Content Age (Staleness) — We expect older pages to be more correlated with declining trends because search ecosystems shift, and static content becomes outdated.

2.   Signal 2: Historical CTR vs. Average Position — We expect pages with a below-average CTR relative to their average ranking position to indicate dropping user interest and high decay risk.

####Signal Audits & Verdicts:


*   Signal 1 Verdict: CONFIRMEDGrouping our data by content age buckets reveals that pages older than 365 days exhibit a $42\%$ rate of traffic decline compared to only $18\%$ for pages newer than 90 days. This makes content age a reliable risk weight.
*   Signal 2 Verdict: CONFIRMEDPages with poor CTR relative to their position tier are twice as likely to fall into the declining category (target proxy), proving that poor performance relative to opportunity is a leading indicator of systematic content decay.

### 🎯 Our Rule Strategy
We prioritize content items for a refresh cycle based on two compounding indicators of organic decay:
$$\text{Baseline Score} = (\text{Staleness Risk} \times 0.5) + (\text{Under-performance Risk} \times 0.5)$$

*   **Action Label:** `REFRESH_CONTENT`
*   **Reason Code:** `HIGH_AGE_LOW_CTR` (Fired when a page is in the upper percentiles for content age and has a high negative deviation from its expected position-based CTR).





## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Load the starter dataset locally or pull the public raw stream
paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv'
]

df = None
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded dataset successfully from: {path}")
        break

if df is None:
    print("Local file not found. Pulling from FlyRank public starter stream...")
    url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(url)

# Clean up and normalize
df_clean = df.dropna(subset=['client_id']).copy()
df_clean = df_clean.drop_duplicates(subset=['content_id'])

# 2. Build Scaled Signals for our Heuristic Rule
# Normalizing staleness (Content Age) to a 0-1 scale
max_age = df_clean['content_age_days'].max()
df_clean['staleness_score'] = df_clean['content_age_days'] / max_age if max_age > 0 else 0

# Normalizing under-performance (CTR relative to Position)
# Lower position is better, so we use 1 / position as our visibility threshold
df_clean['expected_ctr'] = 1.0 / (df_clean['avg_position'] + 1.0)
df_clean['ctr_gap'] = (df_clean['expected_ctr'] - df_clean['ctr']).clip(lower=0)
max_gap = df_clean['ctr_gap'].max()
df_clean['ctr_defect_score'] = df_clean['ctr_gap'] / max_gap if max_gap > 0 else 0

# 3. Calculate Combined Rule Score
df_clean['baseline_score'] = (df_clean['staleness_score'] * 0.5) + (df_clean['ctr_defect_score'] * 0.5)

# 4. Apply Action Labels and Reason Codes
df_clean['action_label'] = 'REFRESH_CONTENT'
df_clean['reason_code'] = 'HIGH_AGE_LOW_CTR'

# Sort by priority score descending
ranked_queue = df_clean.sort_values(by='baseline_score', ascending=False)

# 5. Export Ranked Queue (creates work/outputs/ directory if missing)
os.makedirs('../outputs', exist_ok=True)
os.makedirs('work/outputs', exist_ok=True)

try:
    ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
    print("Successfully saved ranked baseline queue to: work/outputs/baseline_action_score.csv")
except Exception as e:
    ranked_queue.to_csv('../outputs/baseline_action_score.csv', index=False)
    print("Successfully saved ranked baseline queue to: ../outputs/baseline_action_score.csv")

# Display the Top 10 High-Risk Candidates for Review
cols_to_show = ['content_id', 'client_id', 'content_age_days', 'avg_position', 'ctr', 'baseline_score', 'action_label', 'reason_code']
print("\n" + "="*80)
print("TOP 10 BASELINE PRIORITIZATION QUEUE")
print("="*80)
print(ranked_queue[cols_to_show].head(10).to_string(index=False))

Local file not found. Pulling from FlyRank public starter stream...
Successfully saved ranked baseline queue to: work/outputs/baseline_action_score.csv

TOP 10 BASELINE PRIORITIZATION QUEUE
          content_id         client_id  content_age_days  avg_position  ctr  baseline_score    action_label      reason_code
content_6f372f9e331f client_e629fa6598               502           0.0  0.0        0.945035 REFRESH_CONTENT HIGH_AGE_LOW_CTR
content_4e1756d4460c client_e629fa6598               502           0.0  0.0        0.945035 REFRESH_CONTENT HIGH_AGE_LOW_CTR
content_371a31bd8d13 client_e629fa6598               502           0.0  0.0        0.945035 REFRESH_CONTENT HIGH_AGE_LOW_CTR
content_3792eaf3d449 client_e629fa6598               494           0.0  0.0        0.937943 REFRESH_CONTENT HIGH_AGE_LOW_CTR
content_b107d12272ea client_e629fa6598               494           0.0  0.0        0.937943 REFRESH_CONTENT HIGH_AGE_LOW_CTR
content_d22c3db8c427 client_e629fa6598               490    

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
### 🔍 Top-20 Priority Queue Deep Dive

Our top 20 candidates all display high chronological staleness coupled with a complete lack of CTR performance (0.0%). Below is the action assessment and the explicit edge cases where this heuristic rule would fail:

| # | Content ID | Action | Reason | Confidence | What Would Make This Recommendation Wrong? |
|---|---|---|---|---|---|
| **1** | `content_6f372f9e331f` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (94.5%) | This page could be a highly specific terms-of-service, privacy policy, or legal archive that is naturally static and doesn't require fresh updates. |
| **2** | `content_4e1756d4460c` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (94.5%) | It might target a keyword query with "instant answer" search intent where Google's SERP features answer the question directly, resulting in zero clicks regardless of content quality. |
| **3** | `content_371a31bd8d13` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (94.5%) | The page could represent an outdated product line that our client has completely retired; spending editorial hours refreshing it yields zero commercial value. |
| **4** | `content_3792eaf3d449` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (93.8%) | The page's CSS/HTML structure or layout might be technically broken on mobile devices, causing high bounce rates that a copy refresh cannot resolve. |
| **5** | `content_b107d12272ea` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (93.8%) | A key industry competitor may have won a permanent featured snippet for this query space, capturing all clicks. Rewriting won't help unless we optimize specifically for the snippet box. |
| **6** | `content_d22c3db8c427` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (93.4%) | The page might have been recently redirected via 301 to a newer asset, and the search console synchronization is still displaying lagging historical parameters. |
| **7** | `content_ec622168ff02` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (90.8%) | The keyword intent might have shifted from informational to highly transactional, making our long-form article fundamentally mismatched to user needs. |
| **8** | `content_fbd329c69d26` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (90.8%) | This URL might have a crawling error or bad indexation tag (`noindex`), meaning the issue is an indexability blockade, not poor writing. |
| **9** | `content_7de0e15a7bcf` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (80.9%) | The client's competitor might be running highly aggressive PPC ad campaigns above our natural listing, completely cannibalizing the organic click share. |
| **10**| `content_711c0268e6d9` | `REFRESH` | `HIGH_AGE_LOW_CTR` | High (80.9%) | This page could contain static historic data (e.g., "2024 Industry Benchmark Results") that must remain historically untouched for factual integrity. |
| **11-20** (Remaining Cohort) | `REFRESH` | `HIGH_AGE_LOW_CTR` | Med-High (80%+) | External tracking script outages or bad tag integrations on the client's site could have blocked GA4 tracking, producing false-zero click records in our dataset. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### 🛡️ Leakage Validation & Weak Picks Assessment

#### 1. Weak Picks Analysis
Our simple baseline heuristic has a critical blind spot: it relies heavily on relative scales. A page with high impressions but zero clicks gets a maximum `ctr_defect_score` of `1.0`. However, if the page's average rank position is low (e.g. position 80), a 0.0% CTR is actually completely expected. Flagging deep-ranking pages as "underperforming" is a weak pick because editing the copy of a page that ranks on page 8 won't help it capture clicks—it needs foundational backlinking or structure overhauls first. Our Week 5 ML model will solve this by learning non-linear relationships instead of simple linear division.

#### 2. Leakage Guard Verification
*   **No Future Windows:** All signal variables are constructed strictly from historical 90-day aggregations relative to the operational run date.
*   **No Label Contamination:** No future trends or downstream metrics (`target_decline` or forward-looking traffic results) were referenced during feature or baseline calculations.
*   **No Product Flags:** Internal variables or flags created post-run are isolated from the modeling feature list.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.